In [2]:
"""
Arabic OCR Desktop Application
================================
Converts PDF files to images, performs Arabic OCR using EasyOCR,
extracts structured fields, and exports results to Excel.

Requirements: See requirements.txt
Run: python ocr_app.py
"""

import os
import re
import sys
import threading
import tkinter as tk
from tkinter import filedialog, messagebox, ttk

# ─── Lazy imports (heavy libraries loaded only when processing starts) ─────────
# pdf2image, easyocr, pandas, PIL are imported inside the worker thread
# so the GUI launches instantly even if they're slow to import.


# ══════════════════════════════════════════════════════════════════════════════
#  EXTRACTION HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def get_value_before(lines: list[str], keyword: str) -> str:
    """Return the line that appears BEFORE the line containing *keyword*."""
    for i, line in enumerate(lines):
        if keyword in line:
            if i > 0:
                return lines[i - 1].strip()
            elif i < len(lines) - 1:          # fallback: line after keyword
                return lines[i + 1].strip()
    return ""


def parse_birth_info(birth_text: str) -> tuple[str, str]:
    """
    Given a raw OCR line related to birth, return (place, year).
    Place  → first whitespace-separated token.
    Year   → first 4-digit number found with regex.
    """
    place = ""
    year  = ""
    if birth_text:
        parts = birth_text.split()
        if parts:
            place = parts[0]
        match = re.search(r'\d{4}', birth_text)
        if match:
            year = match.group()
    return place, year


def extract_fields(lines: list[str]) -> dict:
    """
    Map OCR text lines to the target fields using keyword search.
    Mirrors the original notebook logic exactly.
    """
    birth_text = get_value_before(lines, "الولادة")
    place, date = parse_birth_info(birth_text)

    return {
        "الرقم الوطني":      get_value_before(lines, "الرقم"),
        "الاسم":             get_value_before(lines, "الاسم"),
        "النسبة":            get_value_before(lines, "النسبة"),
        "اسم الاب":          get_value_before(lines, "الأب"),
        "اسم الجد":          get_value_before(lines, "الجد"),
        "اسم الام ونسبتها":  get_value_before(lines, "الأم"),
        "مكان الولادة":      place,
        "تاريخ الولادة":     date,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  CORE PROCESSING PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def find_poppler_path() -> str | None:
    """
    Try to locate Poppler automatically on common Windows install paths.
    Returns the bin directory path, or None if not found.
    The user can also override this via the GUI entry field.
    """
    common_roots = [
        r"C:\Program Files\poppler",
        r"C:\Program Files (x86)\poppler",
        os.path.join(os.environ.get("USERPROFILE", ""), "Downloads"),
        os.path.join(os.environ.get("LOCALAPPDATA", ""), "poppler"),
    ]
    for root in common_roots:
        if not os.path.isdir(root):
            continue
        # Walk up to 3 levels looking for a 'bin' folder that has pdftoppm.exe
        for dirpath, dirnames, filenames in os.walk(root):
            if "pdftoppm.exe" in filenames:
                return dirpath
    # On Linux/macOS Poppler is usually on PATH – pass None and pdf2image handles it
    return None


def process_pdf(pdf_path: str, reader, poppler_path: str | None,
                log_fn, temp_image_path: str = "temp_ocr_page.png") -> list[dict]:
    """
    Convert one PDF file to images, run OCR on each page, extract fields.

    Returns a list of row-dicts (one per page that has content).
    """
    from pdf2image import convert_from_path
    from PIL import Image, ImageEnhance

    # Patch for older Pillow versions that removed ANTIALIAS
    if not hasattr(Image, "ANTIALIAS"):
        Image.ANTIALIAS = Image.Resampling.LANCZOS

    filename = os.path.basename(pdf_path)
    rows = []

    # ── Convert PDF pages to PIL images ──────────────────────────────────────
    convert_kwargs = dict(pdf_path=pdf_path, dpi=300)
    if poppler_path:
        convert_kwargs["poppler_path"] = poppler_path

    try:
        pages = convert_from_path(**convert_kwargs)
    except Exception as exc:
        log_fn(f"  ✗ Could not convert {filename}: {exc}", error=True)
        return []

    # ── OCR each page ─────────────────────────────────────────────────────────
    for page_num, page_img in enumerate(pages, start=1):
        page_label = f"{filename} – page {page_num}"
        log_fn(f"  OCR: {page_label}")

        try:
            # Contrast enhancement (same as notebook)
            enhancer = ImageEnhance.Contrast(page_img)
            enhanced = enhancer.enhance(1.5)
            enhanced.save(temp_image_path)

            result = reader.readtext(
                temp_image_path,
                detail=1,
                paragraph=False,
                text_threshold=0.1,
                low_text=0.2,
                contrast_ths=0.1,
            )

            lines = [det[1].strip() for det in result if det[1].strip()]

            row = {"اسم الملف": filename}
            row.update(extract_fields(lines))
            rows.append(row)

        except Exception as exc:
            log_fn(f"  ✗ OCR failed on {page_label}: {exc}", error=True)
        finally:
            # Clean up temp file
            if os.path.exists(temp_image_path):
                try:
                    os.remove(temp_image_path)
                except OSError:
                    pass

    return rows


def run_pipeline(input_folder: str, output_folder: str,
                 poppler_path: str | None,
                 log_fn, progress_fn, done_fn):
    """
    Full pipeline: discover PDFs → OCR → save Excel.
    Meant to run in a background thread.
    """
    try:
        import easyocr
        import pandas as pd
    except ImportError as exc:
        log_fn(f"Missing dependency: {exc}\nRun:  pip install {exc.name}", error=True)
        done_fn(success=False)
        return

    # ── Discover PDF files ────────────────────────────────────────────────────
    pdf_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(".pdf")
    ]
    if not pdf_files:
        log_fn("No PDF files found in the selected folder.", error=True)
        done_fn(success=False)
        return

    log_fn(f"Found {len(pdf_files)} PDF file(s). Initialising OCR engine…")

    # ── Initialise EasyOCR (slow, done once) ──────────────────────────────────
    try:
        reader = easyocr.Reader(['ar'], gpu=False)
    except Exception as exc:
        log_fn(f"Failed to load EasyOCR: {exc}", error=True)
        done_fn(success=False)
        return

    log_fn("OCR engine ready. Starting processing…\n")

    all_rows = []

    for idx, filename in enumerate(pdf_files, start=1):
        log_fn(f"[{idx}/{len(pdf_files)}] Processing: {filename}")
        pdf_path = os.path.join(input_folder, filename)
        rows = process_pdf(pdf_path, reader, poppler_path, log_fn)
        all_rows.extend(rows)
        progress_fn(idx / len(pdf_files) * 100)

    # ── Save Excel ────────────────────────────────────────────────────────────
    if not all_rows:
        log_fn("\nNo data was extracted. Excel file not saved.", error=True)
        done_fn(success=False)
        return

    col_order = [
        "اسم الملف", "الرقم الوطني", "الاسم", "النسبة",
        "اسم الاب", "اسم الجد", "اسم الام ونسبتها",
        "مكان الولادة", "تاريخ الولادة",
    ]
    df = pd.DataFrame(all_rows, columns=col_order)

    output_path = os.path.join(output_folder, "output.xlsx")
    try:
        df.to_excel(output_path, index=False)
        log_fn(f"\n✓ Done! Results saved to:\n  {output_path}")
        done_fn(success=True, output_path=output_path)
    except Exception as exc:
        log_fn(f"Failed to save Excel file: {exc}", error=True)
        done_fn(success=False)


# ══════════════════════════════════════════════════════════════════════════════
#  GUI
# ══════════════════════════════════════════════════════════════════════════════

COLORS = {
    "bg":        "#1C1C2E",   # deep navy background
    "panel":     "#252540",   # slightly lighter panel
    "accent":    "#7B61FF",   # electric violet
    "accent2":   "#C084FC",   # soft lavender
    "success":   "#34D399",   # mint green
    "error":     "#F87171",   # coral red
    "text":      "#E2E8F0",   # near-white text
    "muted":     "#94A3B8",   # muted slate
    "border":    "#3B3B5C",   # subtle border
    "btn_hover": "#6147E0",   # darker accent on hover
}

FONT_TITLE  = ("Segoe UI", 18, "bold")
FONT_LABEL  = ("Segoe UI", 10)
FONT_BOLD   = ("Segoe UI", 10, "bold")
FONT_MONO   = ("Consolas", 9)
FONT_BTN    = ("Segoe UI", 11, "bold")


class Tooltip:
    """Simple hover tooltip for any widget."""
    def __init__(self, widget, text):
        self.widget = widget
        self.text   = text
        self.tip    = None
        widget.bind("<Enter>", self._show)
        widget.bind("<Leave>", self._hide)

    def _show(self, _event=None):
        x = self.widget.winfo_rootx() + 20
        y = self.widget.winfo_rooty() + self.widget.winfo_height() + 4
        self.tip = tw = tk.Toplevel(self.widget)
        tw.wm_overrideredirect(True)
        tw.wm_geometry(f"+{x}+{y}")
        lbl = tk.Label(tw, text=self.text, background="#2D2D50",
                       foreground=COLORS["muted"], relief="flat",
                       font=("Segoe UI", 8), padx=6, pady=3)
        lbl.pack()

    def _hide(self, _event=None):
        if self.tip:
            self.tip.destroy()
            self.tip = None


class OCRApp(tk.Tk):
    def __init__(self):
        super().__init__()

        self.title("Arabic OCR Extractor")
        self.resizable(True, True)
        self.minsize(700, 580)
        self.configure(bg=COLORS["bg"])

        # ── State variables ──────────────────────────────────────────────────
        self.input_folder_var  = tk.StringVar()
        self.output_folder_var = tk.StringVar()
        self.poppler_path_var  = tk.StringVar(value=find_poppler_path() or "")
        self.is_running        = False

        self._build_ui()
        self._center_window(700, 640)

    # ── Layout ────────────────────────────────────────────────────────────────

    def _build_ui(self):
        # Title bar area
        header = tk.Frame(self, bg=COLORS["bg"], pady=18)
        header.pack(fill="x", padx=30)

        tk.Label(header, text="🔤  Arabic OCR Extractor",
                 font=FONT_TITLE, bg=COLORS["bg"],
                 fg=COLORS["accent2"]).pack(side="left")

        tk.Label(header,
                 text="PDF → OCR → Excel",
                 font=("Segoe UI", 9), bg=COLORS["bg"],
                 fg=COLORS["muted"]).pack(side="left", padx=(10, 0), pady=(6, 0))

        sep = tk.Frame(self, bg=COLORS["border"], height=1)
        sep.pack(fill="x", padx=20)

        # Main content
        content = tk.Frame(self, bg=COLORS["bg"], padx=30, pady=20)
        content.pack(fill="both", expand=True)

        self._build_folder_row(content, "Input Folder (PDFs):",
                               self.input_folder_var, self._browse_input, required=True)
        self._build_folder_row(content, "Output Folder (Excel):",
                               self.output_folder_var, self._browse_output,
                               placeholder="Leave blank to use Input Folder")
        self._build_poppler_row(content)

        # Progress
        prog_frame = tk.Frame(content, bg=COLORS["bg"])
        prog_frame.pack(fill="x", pady=(18, 6))

        tk.Label(prog_frame, text="Progress", font=FONT_BOLD,
                 bg=COLORS["bg"], fg=COLORS["muted"]).pack(anchor="w")

        style = ttk.Style(self)
        style.theme_use("clam")
        style.configure("Custom.Horizontal.TProgressbar",
                         troughcolor=COLORS["panel"],
                         background=COLORS["accent"],
                         bordercolor=COLORS["border"],
                         lightcolor=COLORS["accent"],
                         darkcolor=COLORS["accent"])

        self.progress_bar = ttk.Progressbar(
            prog_frame, style="Custom.Horizontal.TProgressbar",
            mode="determinate", maximum=100)
        self.progress_bar.pack(fill="x", pady=(4, 0))

        self.progress_label = tk.Label(prog_frame, text="Idle",
                                       font=FONT_MONO, bg=COLORS["bg"],
                                       fg=COLORS["muted"])
        self.progress_label.pack(anchor="e")

        # Log console
        log_frame = tk.Frame(content, bg=COLORS["panel"],
                             bd=0, relief="flat", pady=10, padx=10)
        log_frame.pack(fill="both", expand=True, pady=(6, 0))

        tk.Label(log_frame, text="Log", font=FONT_BOLD,
                 bg=COLORS["panel"], fg=COLORS["muted"]).pack(anchor="w")

        self.log_text = tk.Text(
            log_frame, font=FONT_MONO, bg=COLORS["panel"],
            fg=COLORS["text"], insertbackground=COLORS["accent"],
            relief="flat", bd=0, wrap="word", height=10,
            state="disabled")
        self.log_text.pack(fill="both", expand=True, pady=(4, 0))

        scrollbar = tk.Scrollbar(log_frame, command=self.log_text.yview,
                                 bg=COLORS["panel"], troughcolor=COLORS["panel"],
                                 relief="flat")
        self.log_text["yscrollcommand"] = scrollbar.set

        self.log_text.tag_config("error",   foreground=COLORS["error"])
        self.log_text.tag_config("success", foreground=COLORS["success"])
        self.log_text.tag_config("muted",   foreground=COLORS["muted"])

        # Bottom buttons
        btn_row = tk.Frame(self, bg=COLORS["bg"], pady=14)
        btn_row.pack(fill="x", padx=30)

        self.start_btn = self._make_button(
            btn_row, "▶  Start Processing", self._start, COLORS["accent"])
        self.start_btn.pack(side="left")

        self.clear_btn = self._make_button(
            btn_row, "Clear Log", self._clear_log, COLORS["panel"],
            fg=COLORS["muted"])
        self.clear_btn.pack(side="left", padx=(10, 0))

        self.open_btn = self._make_button(
            btn_row, "📂  Open Output", self._open_output, COLORS["panel"],
            fg=COLORS["success"])
        self.open_btn.pack(side="right")
        self.open_btn.config(state="disabled")

    def _build_folder_row(self, parent, label_text, var,
                          command, required=False, placeholder=""):
        row = tk.Frame(parent, bg=COLORS["bg"], pady=5)
        row.pack(fill="x")

        lbl = tk.Label(row, text=label_text, font=FONT_BOLD,
                       bg=COLORS["bg"], fg=COLORS["text"], width=26, anchor="w")
        lbl.pack(side="left")

        entry = tk.Entry(row, textvariable=var,
                         font=FONT_LABEL, bg=COLORS["panel"],
                         fg=COLORS["text"] if not placeholder else COLORS["muted"],
                         insertbackground=COLORS["accent"],
                         relief="flat", bd=6,
                         highlightthickness=1,
                         highlightbackground=COLORS["border"],
                         highlightcolor=COLORS["accent"])
        entry.pack(side="left", fill="x", expand=True, padx=(0, 8))

        if placeholder and not var.get():
            entry.insert(0, placeholder)
            entry.config(fg=COLORS["muted"])

            def _on_focus_in(_e):
                if entry.get() == placeholder:
                    entry.delete(0, "end")
                    entry.config(fg=COLORS["text"])

            def _on_focus_out(_e):
                if not entry.get():
                    entry.insert(0, placeholder)
                    entry.config(fg=COLORS["muted"])

            entry.bind("<FocusIn>",  _on_focus_in)
            entry.bind("<FocusOut>", _on_focus_out)

        btn = tk.Button(row, text="Browse…", command=command,
                        font=FONT_LABEL, bg=COLORS["border"],
                        fg=COLORS["text"], relief="flat",
                        activebackground=COLORS["accent"],
                        activeforeground="white",
                        padx=10, pady=4, cursor="hand2")
        btn.pack(side="left")

        if required:
            tk.Label(row, text="*", font=FONT_BOLD,
                     bg=COLORS["bg"], fg=COLORS["error"]).pack(side="left", padx=2)

    def _build_poppler_row(self, parent):
        row = tk.Frame(parent, bg=COLORS["bg"], pady=5)
        row.pack(fill="x")

        lbl = tk.Label(row, text="Poppler Bin Path (Windows):",
                       font=FONT_BOLD, bg=COLORS["bg"],
                       fg=COLORS["text"], width=26, anchor="w")
        lbl.pack(side="left")

        entry = tk.Entry(row, textvariable=self.poppler_path_var,
                         font=FONT_LABEL, bg=COLORS["panel"],
                         fg=COLORS["text"],
                         insertbackground=COLORS["accent"],
                         relief="flat", bd=6,
                         highlightthickness=1,
                         highlightbackground=COLORS["border"],
                         highlightcolor=COLORS["accent"])
        entry.pack(side="left", fill="x", expand=True, padx=(0, 8))

        btn = tk.Button(row, text="Browse…",
                        command=self._browse_poppler,
                        font=FONT_LABEL, bg=COLORS["border"],
                        fg=COLORS["text"], relief="flat",
                        activebackground=COLORS["accent"],
                        activeforeground="white",
                        padx=10, pady=4, cursor="hand2")
        btn.pack(side="left")

        note = tk.Label(row, text="ℹ", font=FONT_BOLD,
                        bg=COLORS["bg"], fg=COLORS["accent2"],
                        cursor="hand2")
        note.pack(side="left", padx=4)
        Tooltip(note, "On Linux/macOS Poppler is usually on PATH – leave blank.\n"
                      "On Windows download from: https://github.com/oschwartz10612/poppler-windows/releases")

    def _make_button(self, parent, text, command, bg,
                     fg="white", padx=18, pady=8):
        btn = tk.Button(
            parent, text=text, command=command,
            font=FONT_BTN, bg=bg, fg=fg,
            activebackground=COLORS["btn_hover"],
            activeforeground="white",
            relief="flat", bd=0,
            padx=padx, pady=pady,
            cursor="hand2")
        btn.bind("<Enter>", lambda _: btn.config(bg=COLORS["btn_hover"], fg="white"))
        btn.bind("<Leave>", lambda _: btn.config(bg=bg, fg=fg))
        return btn

    # ── Browse callbacks ───────────────────────────────────────────────────────

    def _browse_input(self):
        path = filedialog.askdirectory(title="Select Input Folder (PDFs)")
        if path:
            self.input_folder_var.set(path)

    def _browse_output(self):
        path = filedialog.askdirectory(title="Select Output Folder")
        if path:
            self.output_folder_var.set(path)

    def _browse_poppler(self):
        path = filedialog.askdirectory(title="Select Poppler bin/ Folder")
        if path:
            self.poppler_path_var.set(path)

    # ── Log helpers ───────────────────────────────────────────────────────────

    def _log(self, message: str, error: bool = False, success: bool = False):
        """Append *message* to the log widget (thread-safe via after())."""
        def _append():
            self.log_text.config(state="normal")
            tag = "error" if error else "success" if success else ""
            self.log_text.insert("end", message + "\n", tag)
            self.log_text.see("end")
            self.log_text.config(state="disabled")
        self.after(0, _append)

    def _clear_log(self):
        self.log_text.config(state="normal")
        self.log_text.delete("1.0", "end")
        self.log_text.config(state="disabled")
        self.progress_bar["value"] = 0
        self.progress_label.config(text="Idle")

    def _set_progress(self, value: float):
        def _update():
            self.progress_bar["value"] = value
            self.progress_label.config(text=f"{value:.0f}%")
        self.after(0, _update)

    # ── Start / Done ──────────────────────────────────────────────────────────

    def _start(self):
        if self.is_running:
            return

        input_folder = self.input_folder_var.get().strip()

        # Validate input folder
        if not input_folder or not os.path.isdir(input_folder):
            messagebox.showerror("Error", "Please select a valid Input Folder.")
            return

        # Determine output folder
        raw_output = self.output_folder_var.get().strip()
        PLACEHOLDER = "Leave blank to use Input Folder"
        if raw_output in ("", PLACEHOLDER):
            output_folder = input_folder
        else:
            if not os.path.isdir(raw_output):
                try:
                    os.makedirs(raw_output, exist_ok=True)
                except OSError as exc:
                    messagebox.showerror("Error",
                                         f"Cannot create output folder:\n{exc}")
                    return
            output_folder = raw_output

        # Poppler path (optional)
        poppler_path = self.poppler_path_var.get().strip() or None
        if poppler_path and not os.path.isdir(poppler_path):
            if not messagebox.askyesno(
                    "Poppler path not found",
                    f"The Poppler path does not exist:\n{poppler_path}\n\n"
                    "Continue anyway? (may fail on Windows)"):
                return

        self.is_running = True
        self.start_btn.config(state="disabled", text="⏳  Processing…")
        self.open_btn.config(state="disabled")
        self.progress_bar["value"] = 0
        self._clear_log()
        self._log("Starting pipeline…", success=True)

        # Store output path for the "Open Output" button
        self._last_output_path = os.path.join(output_folder, "output.xlsx")

        # Run in background thread so GUI stays responsive
        thread = threading.Thread(
            target=run_pipeline,
            args=(input_folder, output_folder, poppler_path,
                  self._log, self._set_progress, self._on_done),
            daemon=True)
        thread.start()

    def _on_done(self, success: bool = False, output_path: str = ""):
        def _update():
            self.is_running = False
            self.start_btn.config(state="normal", text="▶  Start Processing")
            if success:
                self.open_btn.config(state="normal")
                self._set_progress(100)
        self.after(0, _update)

    def _open_output(self):
        path = getattr(self, "_last_output_path", "")
        if path and os.path.isfile(path):
            import subprocess, platform
            if platform.system() == "Windows":
                os.startfile(path)
            elif platform.system() == "Darwin":
                subprocess.call(["open", path])
            else:
                subprocess.call(["xdg-open", path])
        else:
            messagebox.showinfo("Not Found", "Output file not found yet.")

    # ── Window helpers ────────────────────────────────────────────────────────

    def _center_window(self, w: int, h: int):
        self.update_idletasks()
        sw = self.winfo_screenwidth()
        sh = self.winfo_screenheight()
        x  = (sw - w) // 2
        y  = (sh - h) // 2
        self.geometry(f"{w}x{h}+{x}+{y}")


# ══════════════════════════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    app = OCRApp()
    app.mainloop()


In [3]:
! pip install pyinstaller

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -----------------------

In [2]:
! pyinstaller --onefile --windowed --name "ArabicOCR" "C:\\Users\\ASUS\Desktop\\OCR\\ocr_app.ipynb"

1102 INFO: PyInstaller: 6.20.0, contrib hooks: 2026.5
1102 INFO: Python: 3.10.20 (conda)
1114 INFO: Platform: Windows-10-10.0.26200-SP0
1114 INFO: Python environment: C:\Users\ASUS\anaconda3\envs\OCR
1115 INFO: wrote c:\Users\ASUS\Desktop\OCR\ArabicOCR.spec
1145 INFO: Module search paths (PYTHONPATH):
['C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\Scripts\\pyinstaller.exe',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\python310.zip',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\DLLs',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\lib',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\lib\\site-packages',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\lib\\site-packages\\win32',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\lib\\site-packages\\win32\\lib',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\lib\\site-packages\\Pythonwin',
 'C:\\Users\\ASUS\\Desktop\\OCR']
2649 INFO: checking Analysis
2649 INFO: Building Analysis because Analysis-00.toc is non existent
2649 INFO: Looking for Py

In [5]:
!pyinstaller --onefile --windowed --name "ArabicOCR" \
  --hidden-import=easyocr \
  --hidden-import=easyocr.recognition \
  --hidden-import=easyocr.detection \
  --hidden-import=pandas \
  --hidden-import=openpyxl \
  --hidden-import=pdf2image \
  --collect-all=easyocr \
  "C:/Users/ASUS/Desktop/OCR/ocr_app.ipynb"

1215 INFO: PyInstaller: 6.20.0, contrib hooks: 2026.5
1215 INFO: Python: 3.10.20 (conda)
1229 INFO: Platform: Windows-10-10.0.26200-SP0
1229 INFO: Python environment: C:\Users\ASUS\anaconda3\envs\OCR
1231 INFO: wrote c:\Users\ASUS\Desktop\OCR\ArabicOCR.spec
8168 INFO: Module search paths (PYTHONPATH):
['C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\Scripts\\pyinstaller.exe',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\python310.zip',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\DLLs',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\lib',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\lib\\site-packages',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\lib\\site-packages\\win32',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\lib\\site-packages\\win32\\lib',
 'C:\\Users\\ASUS\\anaconda3\\envs\\OCR\\lib\\site-packages\\Pythonwin',
 'C:\\Users\\ASUS\\Desktop\\OCR']
9607 INFO: Appending 'datas' from .spec
9624 INFO: checking Analysis
9627 INFO: Building Analysis because Analysis-00.toc i